# 03a - Distance and Scaling

In [ ]:
# general imports
import numpy as np
import pandas as pd
from IPython.display import display

## Historical Utility Records

The source book supplies the historical case context and field definitions. The notebook loads `Utilities.csv.gz`, distributed by Galit Shmueli, Peter C. Bruce, and Peter Gedeck, directly from the author-maintained [`dmba` repository](https://github.com/gedeck/dmba) at pinned commit [`9b29f4d`](https://github.com/gedeck/dmba/tree/9b29f4dac91fc079113db12b2b9012628d2d0fc8). The repository is MIT licensed. The source values are unchanged, and the notebook strips trailing spaces from display labels in memory.

This analysis selects five fields in a declared order. That selection is part of the representation; it does not imply that every omitted field is irrelevant to every utility comparison.

| Feature | Meaning in this notebook | Recorded unit or scale |
| --- | --- | --- |
| `Cost` | Cost per kilowatt of capacity in place | Source cost per kW |
| `Load_factor` | Annual load factor | Percentage-like source values |
| `Demand_growth` | Peak demand growth from 1974 to 1975 | Percentage-like source values |
| `Sales` | Annual electricity sales | Source sales units |
| `Fuel_Cost` | Total fuel cost | Source cents per kWh |

In [ ]:
# load and lightly clean the data
DATA_URL = (
    "https://raw.githubusercontent.com/gedeck/dmba/"
    "9b29f4dac91fc079113db12b2b9012628d2d0fc8/src/dmba/csvFiles/Utilities.csv.gz"
)

FEATURES = [
    "Cost",
    "Load_factor",
    "Demand_growth",
    "Sales",
    "Fuel_Cost",
]

utilities = pd.read_csv(DATA_URL, compression="gzip")
utilities["Company"] = utilities["Company"].str.strip()
utility_features = utilities.set_index("Company")[FEATURES]

utility_features

The next check confirms the reference-set size, feature order, and missing-value count. The range table exposes the numerical scales without beginning a broad exploratory analysis.

In [ ]:
dataset_check = pd.Series(
    {
        "Utility records": len(utility_features),
        "Selected features": utility_features.shape[1],
        "Missing selected values": int(utility_features.isna().sum().sum()),
    },
    name="Check",
)

feature_ranges = (
    utility_features.max().sub(utility_features.min()).rename("Observed range").to_frame()
)

display(dataset_check.to_frame())
display(feature_ranges)

### Pause: Predict the Raw Driver

Which feature do you expect to exert the most numerical influence on raw Euclidean distances? Use the recorded ranges to explain your prediction without making a claim about substantive importance.

##### Answer

`Sales` is the numerical prediction because its observed range, 14,141, is much larger than the other recorded ranges. This predicts influence under raw distance; it does not establish that sales is the most important business feature.

## Research Question

Which recorded utilities appear most similar to Pacific, and how sensitive is that answer to scaling and distance measure?

## Inspect a Raw Distance

First, let's compare Pacific with the three plausible candidates selected below.

In [ ]:
PACIFIC_CANDIDATES = ["Madison", "Hawaiian", "New England"]


def absolute_gaps(anchor, candidate, values):
    """Return absolute coordinate gaps for one pair of records."""

    return values.loc[anchor].sub(values.loc[candidate]).abs()


def squared_contribution_shares(anchor, candidate, values):
    """Return each coordinate's share of a squared Euclidean sum."""

    squared_gaps = values.loc[anchor].sub(values.loc[candidate]).pow(2)
    return squared_gaps.div(squared_gaps.sum())


raw_gap_table = pd.DataFrame(
    {
        candidate: absolute_gaps("Pacific", candidate, utility_features)
        for candidate in PACIFIC_CANDIDATES
    }
).T

raw_gap_table["Euclidean distance"] = np.sqrt(raw_gap_table[FEATURES].pow(2).sum(axis=1))

display(raw_gap_table.round(3))

This first table shows the absolute coordinate gaps and raw Euclidean distances between Pacific (the "anchor") and each candidate utility.

Madison's `Sales` value differs from Pacific's by only 13 source units. The other two candidates have sales gaps above 300, so raw Euclidean distance places them much farther away.

The next table shows each coordinate's share of the squared sum inside the distance.

In [ ]:
raw_share_table = pd.DataFrame(
    {
        candidate: squared_contribution_shares("Pacific", candidate, utility_features)
        for candidate in PACIFIC_CANDIDATES
    }
).T.mul(100)


display(raw_share_table.round(1).add_suffix(" share (%)"))

The distances between Pacific and the Hawaiian and New England utilities are almost entirely driven by `Sales`. Within the winning Pacific–Madison pair, `Cost` supplies the largest remaining squared contribution.

This distinction matters: a feature can strongly influence which candidate wins by screening out alternatives without being the largest residual contribution inside the winning pair.

## Standardize the Feature Scales

Standardization centers each feature and divides it by its sample standard deviation in this 22-record reference set. Centering applies a common shift and does not change pairwise gaps; division by the feature's standard deviation does. The calculation is explicit so the fitted means and standard deviations remain visible.

$$
z_j = \frac{x_j - \bar{x}_j}{s_j}.
$$

In [ ]:
feature_means = utility_features.mean()
feature_sample_sds = utility_features.std(ddof=1)

standardized_features = utility_features.sub(feature_means).div(feature_sample_sds)

scaling_parameters = pd.DataFrame(
    {
        "Reference mean": feature_means,
        "Sample standard deviation": feature_sample_sds,
    }
)

standardization_check = pd.DataFrame(
    {
        "Standardized mean": standardized_features.mean(),
        "Standardized sample SD": standardized_features.std(ddof=1),
    }
)

display(scaling_parameters.round(3))
display(standardization_check.round(10))

The standardized means are approximately zero, and the sample standard deviations are one. This is a calculation check. A gap of $1$ now represents a one-sample-standard-deviation difference for any feature, so feature gaps are directly comparable numerically. This does not make the feature distributions identical or determine their domain importance.

A later utility must use these same fitted means and standard deviations to remain comparable with this reference set.

## `cdist` for Pairwise Distance Calculation

[`cdist`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cdist.html) computes distances between two collections of observations. Each row is one observation, each column is one feature, and the selected `metric` determines how the coordinate gaps are combined.

The call `cdist(XA, XB, metric=...)` returns a matrix with one row for each observation in `XA` and one column for each observation in `XB`. The documented [`euclidean`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.euclidean.html) and [`cityblock`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cityblock.html) metrics implement Euclidean and Manhattan distance.

The two-dimensional example below uses $A=(0,0)$ and $B=(3,4)$. Euclidean distance follows the straight-line hypotenuse, while Manhattan distance follows the two coordinate-aligned legs.

In [ ]:
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

demo_points = pd.DataFrame(
    {"Feature 1": [0, 3], "Feature 2": [0, 4]},
    index=["A", "B"],
)
point_a = demo_points.loc[["A"]]
point_b = demo_points.loc[["B"]]

display(demo_points)

Calculate distances:

In [ ]:
demo_distances = pd.Series(
    {
        "Euclidean": cdist(point_a, point_b, metric="euclidean").item(),
        "Manhattan": cdist(point_a, point_b, metric="cityblock").item(),
    },
    name="Distance",
)

display(demo_distances.to_frame())

Visualize the result:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot([0, 3], [0, 4], marker="o", linewidth=3, label="Euclidean: 5")
ax.plot(
    [0, 3, 3],
    [0, 0, 4],
    marker="o",
    linewidth=3,
    label="Manhattan: 7",
)

ax.annotate("A", (0, 0), xytext=(-14, -14), textcoords="offset points")
ax.annotate("B", (3, 4), xytext=(8, 4), textcoords="offset points")

ax.set(
    xlabel="Feature 1",
    ylabel="Feature 2",
    xlim=(-0.5, 3.5),
    ylim=(-0.5, 4.5),
    title="Two Ways to Combine the Same Coordinate Gaps",
)

ax.set_aspect("equal")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Compare the Peer Results

To see how standardizing the observations affects similarity, keep the question, records, and five selected features fixed. Then compare raw Euclidean distance with standardized Euclidean distance.

The following code compares the nearest recorded peer for each utility using raw versus standardized Euclidean distance.

In [ ]:
def nearest_peers(values, metric):
    """Return each row's nearest other row and the full distance matrix."""

    distances = cdist(values.to_numpy(), values.to_numpy(), metric=metric)
    # replace the diagonal with infinity to exclude each utility's zero self-distance
    np.fill_diagonal(distances, np.inf)

    nearest_positions = distances.argmin(axis=1)
    nearest = pd.Series(
        values.index.to_numpy()[nearest_positions],
        index=values.index,
        name="Nearest recorded peer",
    )
    distance_frame = pd.DataFrame(
        distances,
        index=values.index,
        columns=values.index,
    )
    return nearest, distance_frame


raw_euclidean_peer, raw_euclidean_distances = nearest_peers(utility_features, metric="euclidean")

standardized_euclidean_peer, standardized_euclidean_distances = nearest_peers(
    standardized_features, metric="euclidean"
)

peer_results = pd.DataFrame(
    {
        "Raw Euclidean": raw_euclidean_peer,
        "Standardized Euclidean": standardized_euclidean_peer,
    }
)

peer_results["Scale changed peer"] = peer_results["Raw Euclidean"].ne(
    peer_results["Standardized Euclidean"]
)

display(peer_results.loc[["Pacific", "Arizona", "Idaho", "Nevada", "Texas"]])

print(
    f"Scale changed the nearest peer for "
    f"{peer_results['Scale changed peer'].sum()} of {len(peer_results)} anchors."
)

Pacific's nearest recorded peer changes from Madison to Hawaiian. Across all 22 anchors, raw and standardized Euclidean distance select different peers for 19 utilities. That count describes these records, features, and procedures; it is not a population estimate.

Inspect the leading standardized Euclidean candidates for Pacific before treating Hawaiian as a decisive recommendation.

In [ ]:
pacific_standardized_candidates = (
    standardized_euclidean_distances.loc["Pacific"]
    .sort_values()
    .head(5)
    .rename("Standardized Euclidean distance")
    .to_frame()
)

pacific_standardized_candidates.round(3)

Hawaiian's distance is approximately 1.548, while New England's is approximately 1.552. The near tie makes the peer identity sensitive even within the declared standardized Euclidean procedure.

### Why Sensitivity Rather Than Significance?

These distances are deterministic summaries of one fixed historical reference set. Each utility is represented by one aggregate five-feature vector. Without repeated measurements or a declared model for sampling or measurement uncertainty, the difference between two distances has no defensible sampling distribution, so a significance test is not meaningful here. The near tie is a small numerical margin, not a p-value or evidence of statistical equivalence.

Instead, this notebook evaluates sensitivity: does the selected peer change when reasonable analytical choices such as feature scaling or distance measure change? With repeated measurements or a credible uncertainty model, resampling could address a separate question about statistical uncertainty.

## Hold Scale Fixed and Change the Metric

Now keep the standardized representation fixed and change only the metric from Euclidean distance, $L_2$, to Manhattan distance, $L_1$. This isolates the consequence of the aggregation rule from the consequence of scaling.

In [ ]:
standardized_manhattan_peer, standardized_manhattan_distances = nearest_peers(
    standardized_features, metric="cityblock"
)

peer_results["Standardized Manhattan"] = standardized_manhattan_peer
peer_results["Metric changed peer"] = peer_results["Standardized Euclidean"].ne(
    peer_results["Standardized Manhattan"]
)

metric_changes = peer_results.loc[
    peer_results["Metric changed peer"],
    ["Standardized Euclidean", "Standardized Manhattan"],
]

display(metric_changes)
print(
    f"Metric choice changed the nearest peer for "
    f"{len(metric_changes)} of {len(peer_results)} anchors."
)

Pacific's nearest recorded peer changes from Hawaiian under standardized Euclidean distance to New England under standardized Manhattan distance. Five of the 22 anchors change peer when scale remains fixed and only the metric changes.

## Explain One Changed Peer

Set `ASSIGNED_ANCHOR` to the utility assigned to your pair or group: `Arizona`, `Idaho`, `Nevada`, or `Texas`. Do not change the feature set or search for an anchor that produces a preferred result.

For a Euclidean comparison, the contribution share from feature $j$ is

$$
\frac{(x_j-y_j)^2}{\sum_k(x_k-y_k)^2}.
$$

The share explains the numerical composition of one declared pairwise distance. It is not a measure of the feature's substantive importance.

In [ ]:
ASSIGNED_ANCHOR = "Arizona"
VALID_ASSIGNED_ANCHORS = {"Arizona", "Idaho", "Nevada", "Texas"}

if ASSIGNED_ANCHOR not in VALID_ASSIGNED_ANCHORS:
    raise ValueError(f"Choose one of {sorted(VALID_ASSIGNED_ANCHORS)}.")

raw_assigned_peer = raw_euclidean_peer.loc[ASSIGNED_ANCHOR]
standardized_assigned_peer = standardized_euclidean_peer.loc[ASSIGNED_ANCHOR]

assigned_profile = pd.DataFrame(
    {
        f"Raw → {raw_assigned_peer}": squared_contribution_shares(
            ASSIGNED_ANCHOR,
            raw_assigned_peer,
            utility_features,
        ),
        f"Standardized → {standardized_assigned_peer}": (
            squared_contribution_shares(
                ASSIGNED_ANCHOR,
                standardized_assigned_peer,
                standardized_features,
            )
        ),
    }
).mul(100)

assigned_profile.index.name = "Feature"

display(
    pd.Series(
        {
            "Assigned anchor": ASSIGNED_ANCHOR,
            "Raw Euclidean peer": raw_assigned_peer,
            "Standardized Euclidean peer": standardized_assigned_peer,
        },
        name="Result",
    ).to_frame()
)

display(assigned_profile.round(1).add_suffix(" share (%)"))

In [ ]:
assigned_candidates = pd.Index([raw_assigned_peer, standardized_assigned_peer]).unique()

assigned_distance_comparison = pd.DataFrame(
    {
        "Raw Euclidean": raw_euclidean_distances.loc[ASSIGNED_ANCHOR, assigned_candidates],
        "Standardized Euclidean": standardized_euclidean_distances.loc[
            ASSIGNED_ANCHOR, assigned_candidates
        ],
    }
)

assigned_distance_comparison.index.name = "Candidate"

display(assigned_distance_comparison.round(3))

The distance table evaluates both peer candidates under both representations. It establishes the ranking reversal: the raw peer has the smaller raw distance, while the standardized peer has the smaller standardized distance. The contribution table answers a different question by showing how the features compose each winning pair's squared distance. Use both tables to explain the changed peer without treating a numerical contribution share as substantive importance.

### Pause: Report the Mechanism

Identify the dominant raw contribution. Then describe how standardization changes the contribution profile. Prepare two sentences: one sentence about the recorded result and one bounded interpretation.

##### Answer

With the default Arizona anchor, Central is closer than Commonwealth in raw units: their Euclidean distances from Arizona are 140.265 and 2,654.055. After standardization, Commonwealth is closer: its distance is 1.024, compared with 1.372 for Central. `Sales` supplies 92.6% of the raw Arizona–Central squared distance. In the standardized Arizona–Commonwealth comparison, the contribution is distributed across `Sales` (53.3%), `Demand_growth` (16.6%), `Cost` (16.2%), `Load_factor` (12.3%), and `Fuel_Cost` (1.6%).

The changed peer is consistent with raw numerical scale controlling the original geometry. It does not establish that Commonwealth is Arizona's objectively correct or operationally interchangeable peer.

## Record the Sensitivity Evidence

The final record should connect the decision-facing answer with the mechanism and the limits of the comparison.

In [ ]:
evidence_summary = pd.Series(
    {
        "Pacific raw Euclidean peer": raw_euclidean_peer.loc["Pacific"],
        "Pacific standardized Euclidean peer": (standardized_euclidean_peer.loc["Pacific"]),
        "Pacific standardized Manhattan peer": (standardized_manhattan_peer.loc["Pacific"]),
        "Anchors changed by scaling": (
            f"{peer_results['Scale changed peer'].sum()} of {len(peer_results)}"
        ),
        "Anchors changed by metric": (
            f"{peer_results['Metric changed peer'].sum()} of {len(peer_results)}"
        ),
    },
    name="Recorded evidence",
)

evidence_summary.to_frame()

### Pause: Write the Bounded Conclusion

Write a short record entry that answers five questions:

1. What operational question and five-feature representation did this notebook declare?
2. What changed when the features were standardized?
3. What changed when scale stayed fixed and $L_2$ changed to $L_1$?
4. What did the assigned-anchor contribution profile reveal about the mechanism?
5. What conclusion is supported, and what stronger conclusion is not supported?

##### Answer

For a first-pass peer to support model transfer or benchmarking, this notebook compared 22 historical utilities using `Cost`, `Load_factor`, `Demand_growth`, `Sales`, and `Fuel_Cost`. Pacific's nearest recorded peer changed from Madison under raw Euclidean distance to Hawaiian after standardization; 19 of the 22 anchors changed peer under that scale comparison.

With standardization fixed, Pacific's peer changed to New England under Manhattan distance, and 5 of the 22 anchors changed peer. The assigned-anchor profile showed how raw numerical scale can concentrate the squared sum and how standardization redistributes numerical influence. These results support treating the peer identity as representation-sensitive in this historical case. They do not identify one true peer, estimate a population rate, or show that standardization is always required.

## Closing Perspective

Similarity is not an inherent property of two records. It results from the representation, scaling, and distance measure chosen for a stated purpose. Before acting on a nearest peer, record those choices and check whether the conclusion survives reasonable alternatives.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).